# LeaseGuard Phase 6 — Compare T4 baselines

This notebook does **not** need a GPU. It reads the three candidate run files from Drive and selects the strongest practical unmodified base model using a lexicographic order: schema validity, extraction F1, answer status accuracy, evidence recall, speed, then lower peak VRAM.

It will not invent a blended LeaseGuard score. Official professional-benchmark claims still require LegalBench, CUAD, ContractNLI, or LegalBench-RAG with pinned evaluators.


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/regional-specter/LeaseGuard.git"
REPO_DIR = Path("/content/LeaseGuard")
DRIVE_ROOT = Path("/content/drive/MyDrive/leaseguard")
REPORT_DIR = DRIVE_ROOT / "reports" / "baselines"
OUTPUT = REPORT_DIR / "phase6_comparison.json"

print({"drive": str(DRIVE_ROOT), "output": str(OUTPUT)})

In [ ]:
import sys
import sysconfig
from subprocess import run

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab; using paths already set above.")

if not (REPO_DIR / "src" / "leaseguard").is_dir():
    empty = REPO_DIR.exists() and REPO_DIR.is_dir() and not any(REPO_DIR.iterdir())
    if empty:
        REPO_DIR.rmdir()
    drive_repo = DRIVE_ROOT / "repo"
    if (drive_repo / "src" / "leaseguard").is_dir() and not REPO_DIR.exists():
        os.symlink(drive_repo, REPO_DIR)
    else:
        clone = run(["git", "clone", REPO_URL, str(REPO_DIR)])
        if clone.returncode != 0:
            raise SystemExit(
                "git clone failed; set REPO_URL or copy the repo to Drive/leaseguard/repo."
            )

os.chdir(REPO_DIR)
src = str(REPO_DIR / "src")
if src not in sys.path:
    sys.path.insert(0, src)


def _pip(*packages: str) -> int:
    extra: list[str] = []
    marker = Path(sysconfig.get_path("stdlib")) / "EXTERNALLY-MANAGED"
    if marker.is_file():
        extra.append("--break-system-packages")
    cmd = [sys.executable, "-m", "pip", "install", *extra, *packages]
    print("+", " ".join(cmd))
    return run(cmd).returncode


try:
    from leaseguard.colab import install_runtime
except ImportError:
    install_runtime = None

if install_runtime is not None:
    install_runtime(REPO_DIR, inference=False)
elif _pip("-e", ".") != 0:
    print("editable install failed; using src/ and runtime deps")
    status = _pip("pydantic>=2.11,<3", "pypdf>=5.1,<6", "python-docx>=1.1,<2")
    if status != 0:
        raise SystemExit("runtime dependency install failed")

In [ ]:
from leaseguard.evaluation.cli import main as evaluation_main

runs = sorted(REPORT_DIR.glob("*-schema-full-4bit.json"))
print("required runs found:", [path.name for path in runs])
if not runs:
    raise SystemExit(
        "No schema-full-4bit runs in Drive. Finish the three candidate notebooks first."
    )

argv = ["compare-baselines", *[str(path) for path in runs], "--output", str(OUTPUT)]
status = evaluation_main(argv)
raise SystemExit(status)